# 28c · CXR-LT — shard **letterbox @1022** (giữ 100% FOV)

`19_fov_ablation.ipynb` đo được: **28.5% diện tích ảnh gốc không bao giờ tới được model**, trong
đó **16.8% trục dài mất vĩnh viễn** ngay lúc tạo shard. Nguyên nhân là `04a` scale theo **cạnh
ngắn** rồi cắt vuông. Verdict của nb19 là `UU TIEN` — nhưng F3 chưa từng chạy.

Notebook này dựng bộ shard **letterbox** cho toàn bộ **377,110 ảnh** CXR-LT: scale theo **cạnh
dài** rồi **đệm** cho vuông. Không cắt gì cả.

```
04a  (crop)      scale canh NGAN -> 559, cat vuong, roi CenterCrop(518)
                 => model chi thay 71.5% khung, o 0.626 mm/pixel
28c  (letterbox) scale canh DAI  -> 1022, dem cho vuong
                 => model thay 100% khung, o 0.411 mm/pixel
```

## Vì sao **1022**, và vì sao letterbox thắng cả hai trục

Với ảnh trung vị MIMIC-CXR `2544×3056` (`AR` 1.20), so ba lựa chọn:

| pipeline | FOV | mm/pixel | GPU |
|---|---|---|---|
| `04a` crop 559 → CenterCrop(518) — **hiện tại** | 71.5% | 0.626 | 1.0× |
| crop @1022 | 83.2% | 0.342 | 6.5× |
| **letterbox @1022** | **100%** | **0.411** | 6.5× |

Hai điều rút ra:

**Crop mất cùng một *tỉ lệ* khung ở MỌI độ phân giải.** FOV của crop là `2544/3056 = 83.2%`, do
**tỉ lệ khung ảnh** quyết định, không do kích thước đầu vào. Nâng độ phân giải không lấy lại một
pixel ngoại vi nào — chỉ làm phần giữ lại nét hơn.

**Pipeline hiện tại còn tệ hơn crop thường** (71.5% so với 83.2%). Chênh 11.7 điểm phần trăm đến
hoàn toàn từ việc lưu ở 559 để chừa headroom rồi `CenterCrop(518)` cắt lần hai — headroom đó dành
cho `RandomResizedCrop` lúc train, mà Stage 3 là probe trên đặc trưng **đông cứng**, không augment.

`letterbox @1022` **thắng pipeline hiện tại trên cả hai trục cùng lúc**: 100% FOV thay vì 71.5%,
và 0.411 mm/pixel thay vì 0.626. Không phải đánh đổi.

`1022 = 73 × 14`, chia hết cho patch size. Positional embedding của RAD-DINO sẽ được **nội suy tự
động** từ lưới 37×37 lên 73×73 (`Dinov2Embeddings.interpolate_pos_encoding`, không cần cờ nào).
Đó là kéo giãn 2× — DINOv2 chịu được, nhưng **vẫn là lệch phân bố và phải đo**, không được giả định.

`31_cxrlt_embed.ipynb` nạp bộ này **không crop**. Từ 1022 hạ xuống 518 lúc nạp là ra arm
letterbox@518, nên bộ này **bao trùm** mọi mức thấp hơn.

> **Không đụng vào bất kỳ shard nào đang có.** Ghi sang thư mục riêng `cxr_1022_letterbox_cxrlt/`.

**CPU + Drive I/O, không cần GPU.** 378 shard, **~62 GB**.

Ở 1022px mỗi ảnh tốn nhiều CPU hơn, nhưng hai công tắc tăng tốc bù lại:

| | tiết kiệm | vì sao |
|---|---|---|
| `FAST_DECODE` (`Image.draft`) | **~1.7×** | libjpeg giải mã thẳng ở 1/2 trong miền DCT thay vì bung đủ 7.8 Mpixel rồi vứt 89%. Không bao giờ xuống dưới `LB_SIZE`, nên BICUBIC vẫn làm phần hạ mẫu cuối |
| `JPEG_OPTIMIZE = False` | **~1.15×** | `optimize=True` chạy thêm một lượt tối ưu Huffman, chậm gần gấp đôi, đổi lại file nhỏ hơn ~5% |
| chia 2 runtime | **~2×** | mỗi VM có CPU riêng |

Không tăng tốc: ~4.5–6 giờ. Bật cả hai: **~2.5–3.5 giờ**. Thêm hai runtime: **~1.5 giờ**.

> **`FAST_DECODE` mặc định TẮT.** Chạy **C4-bis** trước — nó đo chênh lệch pixel và tốc độ thật
> giữa hai đường giải mã **trên chính ảnh của bạn**, rồi mới quyết. Cổng chấm trên ảnh **tệ
> nhất** chứ không phải trung bình, vì một ảnh hỏng vẫn là hỏng.
>
> `JPEG_OPTIMIZE = False` thì bật sẵn — nó chỉ đổi cách mã hoá Huffman, không đụng tới pixel.

Thanh tiến trình cập nhật **theo từng ảnh**, có `img/s`, đã chạy bao lâu và còn bao lâu.

## C0 · Config

In [1]:
import os, io, sys, time, tarfile, json
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

# ---- giong 04a ----
DRIVE_IMG  = '/content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org/files'
SHARD_SIZE = 1000
GRAYSCALE  = True
JPEG_Q     = 90
N_THREADS  = 32

# ---- KHAC 04a: khong headroom, khong crop, va CAO HON kich thuoc model ----
# 04a luu 559 de chua headroom roi CenterCrop(518) -> xen lai 7.3% moi truc.
# O day luu thang o LB_SIZE va nap KHONG crop, nen khong mat gi them.
LB_SIZE    = 1022       # = 73 x 14. Xem phan mo dau: thang pipeline hien tai ca FOV lan mm/pixel
PAD_VALUE  = 0          # vien den — cung mau voi vien collimation san co tren phim

# ---- tang toc; C3-bis DO chenh lech truoc khi tin ----
FAST_DECODE   = False   # <<< MAC DINH TAT. Chay C4-bis TRUOC roi moi bat.
                        # Image.draft(): libjpeg giai ma thang o 1/2 hoac 1/4 trong mien DCT.
                        # KHONG BAO GIO xuong duoi LB_SIZE — no chon ty le lon nhat con >= dich,
                        # nen BICUBIC van lam phan ha mau cuoi. Tiet kiem ~75% chi phi giai ma.
JPEG_OPTIMIZE = False   # optimize=True chay them mot luot toi uu Huffman: cham gan gap doi,
                        # doi lai file nho hon ~5%. Khong dang o day.

OUT_DIR   = os.path.join(DATA_DIR, 'cxrlt_2024')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
os.makedirs(SHARD_DIR, exist_ok=True)

SHARD_FROM = None       # None -> 0        | chia cho 2 runtime: A=(0, N//2), B=(N//2, None)
SHARD_TO   = None       # None -> N_SHARDS

P_IDX = os.path.join(OUT_DIR, 'cxrlt_index.parquet')
assert os.path.exists(P_IDX), f'khong thay {P_IDX} -> chay 28_cxrlt_inventory.ipynb truoc'
assert os.path.isdir(DRIVE_IMG), f'khong thay anh goc: {DRIVE_IMG}'
print(f'anh goc  : {DRIVE_IMG}')
print(f'shard moi: {SHARD_DIR}')

Mounted at /content/drive
anh goc  : /content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org/files
shard moi: /content/drive/MyDrive/MIMIC_CXR/planB_out/cxr_1022_letterbox_cxrlt


## C1 · `MODEL_SIZE` từ processor + cảnh báo nội suy positional embedding

In [2]:
from transformers import AutoImageProcessor
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = (proc.crop_size['height'] if getattr(proc, 'do_center_crop', False)
              else (proc.size.get('shortest_edge') or proc.size.get('height')))
PATCH = 14
assert LB_SIZE % PATCH == 0, f'LB_SIZE={LB_SIZE} khong chia het cho patch {PATCH}'
_g = LB_SIZE // PATCH
print(f'MODEL_SIZE (RAD-DINO huan luyen o day) : {MODEL_SIZE}  -> {MODEL_SIZE//PATCH}x{MODEL_SIZE//PATCH} patch')
print(f'LB_SIZE    (ta luu o day)              : {LB_SIZE}  -> {_g}x{_g} = {_g*_g:,} patch')
if LB_SIZE != MODEL_SIZE:
    print(f'\n!! LB_SIZE != MODEL_SIZE: positional embedding se duoc NOI SUY '
          f'{MODEL_SIZE//PATCH}x{MODEL_SIZE//PATCH} -> {_g}x{_g} (keo gian {_g/(MODEL_SIZE//PATCH):.2f}x)')
    print('   Dinov2Embeddings tu lam viec nay, khong can co. Nhung day LA lech phan bo')
    print('   so voi luc huan luyen -> phai DO o Stage 3, khong duoc gia dinh la tot.')
    print(f'   Chi phi attention ~{(_g*_g/(MODEL_SIZE//PATCH)**2)**2:.1f}x o phan bac hai.')
print('\nnen: khong CenterCrop o 31_cxrlt_embed.ipynb cho bo nay')

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

MODEL_SIZE (RAD-DINO huan luyen o day) : 518  -> 37x37 patch
LB_SIZE    (ta luu o day)              : 1022  -> 73x73 = 5,329 patch

!! LB_SIZE != MODEL_SIZE: positional embedding se duoc NOI SUY 37x37 -> 73x73 (keo gian 1.97x)
   Dinov2Embeddings tu lam viec nay, khong can co. Nhung day LA lech phan bo
   so voi luc huan luyen -> phai DO o Stage 3, khong duoc gia dinh la tot.
   Chi phi attention ~15.2x o phan bac hai.

nen: khong CenterCrop o 31_cxrlt_embed.ipynb cho bo nay


## C2 · Danh sách ảnh — **toàn bộ** CXR-LT

Khác `28b`: bộ này dựng lại **cả 377,110 ảnh**, không chỉ phần thiếu. Bộ crop cũ không dùng lại
được vì nó đã bị cắt mất phần rìa — thông tin đó chỉ còn trong ảnh gốc.

In [3]:
IDX = pd.read_parquet(P_IDX, columns=['dicom_id', 'study_id', 'subject_id',
                                      'split', 'view_group'])
work = (IDX.sort_values(['subject_id', 'study_id', 'dicom_id']).reset_index(drop=True))
work['shard'] = np.arange(len(work)) // SHARD_SIZE
N_SHARDS = int(work.shard.max()) + 1
print(f'{len(work):,} anh -> {N_SHARDS} shard x toi da {SHARD_SIZE}')
print(f'theo split: {work.split.value_counts().to_dict()}')
print(f'uoc tinh  : {len(work)/22/3600:.1f}-{len(work)/18/3600:.1f} gio '
      f'(28b do 24 img/s o 559px; o {LB_SIZE}px cham hon vi nen JPEG to hon)')
print(f'dung luong: ~{len(work)*42*(LB_SIZE/518)**2/1e6:.0f} GB '
      f'(42 KB/anh o 518px, ty le theo dien tich)')

_r = work.iloc[0]; _s = str(int(_r.subject_id))
_rel = f'p{_s[:2]}/p{_s}/s{int(_r.study_id)}/{_r.dicom_id}.jpg'
assert os.path.exists(os.path.join(DRIVE_IMG, _rel)), \
    f'anh goc dau tien khong ton tai: {os.path.join(DRIVE_IMG, _rel)}'
print(f'\nmau duong dan OK: .../{_rel}')

377,110 anh -> 378 shard x toi da 1000
theo split: {'train': 258871, 'test': 78946, 'val': 39293}
uoc tinh  : 4.8-5.8 gio (28b do 24 img/s o 559px; o 1022px cham hon vi nen JPEG to hon)
dung luong: ~62 GB (42 KB/anh o 518px, ty le theo dien tich)

mau duong dan OK: .../p10/p10000032/s50414267/02aa804e-bde0afdd-112c0b34-7bc16630-4e384014.jpg


## C3 · Kiểm `_letterbox` trên ảnh thật trước khi chạy 2 giờ

Ba tính chất phải đúng, kiểm trên vài ảnh gốc có tỉ lệ khung khác nhau:

1. đầu ra đúng `LB_SIZE × LB_SIZE`
2. **tỉ lệ khung được giữ** — vật thể không bị bóp méo
3. **không mất pixel nào**: mọi pixel gốc đều ánh xạ được vào ảnh mới

In [4]:
def _letterbox(im, size, pad=PAD_VALUE):
    '''Scale theo canh DAI roi dem cho vuong. Giu nguyen ty le khung, khong cat gi.'''
    w, h = im.size
    s = size / max(w, h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    im = im.resize((nw, nh), Image.BICUBIC)
    out = Image.new(im.mode, (size, size), pad)
    out.paste(im, ((size - nw) // 2, (size - nh) // 2))     # can giua
    return out

# --- kiem tren anh THAT, lay 6 anh co AR khac nhau ---
_probe = work.sample(min(6, len(work)), random_state=0)
print(f'{"goc":>12}  {"AR":>5}  {"sau letterbox":>15}  {"vung anh":>13}  giu FOV')
for r in _probe.itertuples():
    _s = str(int(r.subject_id))
    p = os.path.join(DRIVE_IMG, f'p{_s[:2]}/p{_s}/s{int(r.study_id)}/{r.dicom_id}.jpg')
    im0 = Image.open(p).convert('L' if GRAYSCALE else 'RGB')
    w, h = im0.size
    lb = _letterbox(im0, LB_SIZE)
    sc = LB_SIZE / max(w, h)
    nw, nh = max(1, round(w*sc)), max(1, round(h*sc))
    assert lb.size == (LB_SIZE, LB_SIZE), lb.size
    # ty le khung giu duoc? so AR truoc/sau tren vung anh (bo phan dem)
    assert abs((nw/nh) - (w/h)) < 0.01, f'ty le khung bi bop meo: {w}x{h} -> {nw}x{nh}'
    print(f'{w:>5}x{h:<6}  {max(w,h)/min(w,h):>5.2f}  {LB_SIZE}x{LB_SIZE:<10}  '
          f'{nw:>5}x{nh:<6}  100%')

# --- so truc tiep voi cach cua 04a de thay phan mat ---
_w, _h = 2544, 3056                                        # trung vi cua MIMIC-CXR (nb19)
_crop_keep = (518/559)**2 / (max(_w,_h)/min(_w,_h))
print(f'\nvoi anh trung vi {_w}x{_h} (AR {max(_w,_h)/min(_w,_h):.2f}):')
print(f'  04a crop  : model nhin thay {_crop_keep:.1%} dien tich goc')
print(f'  28c letterbox: model nhin thay 100.0% dien tich goc')
print(f'  -> doi lai: do phan giai tren moi pixel giam '
      f'{(518/max(_w,_h))/(559/min(_w,_h)):.2f}x')

         goc     AR    sau letterbox       vung anh  giu FOV
 2539x3050     1.20  1022x1022          851x1022    100%
 1736x2022     1.16  1022x1022          877x1022    100%
 1760x2140     1.22  1022x1022          841x1022    100%
 2544x3056     1.20  1022x1022          851x1022    100%
 3050x2539     1.20  1022x1022         1022x851     100%
 2978x2539     1.17  1022x1022         1022x871     100%

voi anh trung vi 2544x3056 (AR 1.20):
  04a crop  : model nhin thay 71.5% dien tich goc
  28c letterbox: model nhin thay 100.0% dien tich goc
  -> doi lai: do phan giai tren moi pixel giam 0.77x


## C4 · Hàm dựng shard

In [5]:
def _read_bytes(sub, st, dic):
    s = str(int(sub)); rel = f'p{s[:2]}/p{s}/s{int(st)}/{dic}.jpg'
    with open(os.path.join(DRIVE_IMG, rel), 'rb') as f:
        return f.read()

def _prep(raw):
    im = Image.open(io.BytesIO(raw))
    if FAST_DECODE:
        # chon ty le DCT lon nhat ma ket qua VAN >= LB_SIZE -> khong mat gi duoi dich
        im.draft('L' if GRAYSCALE else 'RGB', (LB_SIZE, LB_SIZE))
    im = im.convert('L') if GRAYSCALE else im.convert('RGB')
    im = _letterbox(im, LB_SIZE)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=JPEG_Q, optimize=JPEG_OPTIMIZE)
    return buf.getvalue()

def _one(r):
    try:
        return r.dicom_id, _prep(_read_bytes(r.subject_id, r.study_id, r.dicom_id)), None
    except Exception as e:
        return r.dicom_id, None, f'{type(e).__name__}: {e}'

def build_shard(i, pbar=None):
    final = os.path.join(SHARD_DIR, f'lb_{i:05d}.tar')
    if os.path.exists(final): return 'skip', 0, []
    tmp  = final + '.tmp'                     # rename-on-complete
    rows = work[work.shard == i]
    errs = []
    with tarfile.open(tmp, 'w') as tar, ThreadPoolExecutor(N_THREADS) as ex:
        for dic, blob, err in ex.map(_one, [r for r in rows.itertuples()]):
            if pbar is not None: pbar.update(1)          # cap nhat TUNG ANH
            if err: errs.append((dic, err)); continue
            info = tarfile.TarInfo(f'{dic}.jpg'); info.size = len(blob)
            tar.addfile(info, io.BytesIO(blob))
    os.replace(tmp, final)
    return 'done', len(rows) - len(errs), errs

## C4-bis · `draft()` làm mất bao nhiêu? — **đo, không đoán**

`FAST_DECODE` đổi đường giải mã, nên phải biết nó đổi *pixel* bao nhiêu trước khi bỏ 5 giờ vào.

Cell này lấy vài chục ảnh thật, chạy **cả hai đường** — giải mã đầy đủ và `draft` — rồi so pixel:
sai lệch lớn nhất, sai lệch trung bình, PSNR. Kèm đo tốc độ thật của từng đường.

Nếu PSNR > 45 dB thì chênh lệch nằm dưới mức nhiễu nén JPEG q90 và có thể bật `FAST_DECODE` yên
tâm. Nếu thấp hơn, đặt `FAST_DECODE = False` và chấp nhận chạy lâu hơn.

In [6]:
import time as _t

def _prep_ref(raw):                       # duong DAY DU, khong draft
    im = Image.open(io.BytesIO(raw))
    im = im.convert('L') if GRAYSCALE else im.convert('RGB')
    im = _letterbox(im, LB_SIZE)
    b = io.BytesIO(); im.save(b, 'JPEG', quality=JPEG_Q, optimize=JPEG_OPTIMIZE)
    return b.getvalue()

_smp = work.sample(min(24, len(work)), random_state=7)
_raws, _scales = [], []
for r in _smp.itertuples():
    _raws.append(_read_bytes(r.subject_id, r.study_id, r.dicom_id))
for raw in _raws[:6]:                     # ty le DCT ma draft chon
    _im = Image.open(io.BytesIO(raw)); _w0, _h0 = _im.size
    _im.draft('L', (LB_SIZE, LB_SIZE))
    _scales.append(f'{_w0}x{_h0} -> {_im.size[0]}x{_im.size[1]}')
print('draft chon ty le (vai vi du):')
for x in _scales: print(f'   {x}')

_t0 = _t.time(); [_prep_ref(r) for r in _raws]; _t_ref = (_t.time()-_t0)/len(_raws)
_t0 = _t.time(); [_prep(r)     for r in _raws]; _t_fast = (_t.time()-_t0)/len(_raws)
print(f'\ntoc do 1 luong: day du {_t_ref*1000:5.1f} ms/anh | draft {_t_fast*1000:5.1f} ms/anh'
      f'  -> nhanh {_t_ref/max(_t_fast,1e-9):.2f}x')

_d = []
for raw in _raws:
    a = np.asarray(Image.open(io.BytesIO(_prep_ref(raw))), dtype='float32')
    b = np.asarray(Image.open(io.BytesIO(_prep(raw))),     dtype='float32')
    _d.append((np.abs(a-b).max(), np.abs(a-b).mean(),
               10*np.log10(255**2/max(((a-b)**2).mean(), 1e-9))))
_d = np.array(_d)
print(f'\nchenh lech pixel tren {len(_raws)} anh that (thang 0-255):')
print(f'   sai lech LON NHAT   : {_d[:,0].max():6.1f}')
print(f'   sai lech trung binh : {_d[:,1].mean():6.2f}')
print(f'   PSNR                : {_d[:,2].mean():6.1f} dB  (thap nhat {_d[:,2].min():.1f})')
_ok = _d[:,2].min() > 45          # cham tren anh TE NHAT, khong phai trung binh
if _ok:
    print(f'   -> anh te nhat van {_d[:,2].min():.1f} dB > 45: duoi muc nhieu nen q90.')
    print('      Dat FAST_DECODE = True o C0 roi chay lai tu C0.')
else:
    print(f'   -> co anh xuong {_d[:,2].min():.1f} dB < 45: GIU FAST_DECODE = False.')
    print('      Mat them ~1.5x thoi gian nhung khong doi duong giai ma.')
print(f'\nFAST_DECODE = {FAST_DECODE} | JPEG_OPTIMIZE = {JPEG_OPTIMIZE}')

draft chon ty le (vai vi du):
   2544x3056 -> 1272x1528
   2544x3056 -> 1272x1528
   2140x1760 -> 2140x1760
   2544x3048 -> 1272x1524
   2544x3056 -> 1272x1528
   1530x1846 -> 1530x1846

toc do 1 luong: day du  54.3 ms/anh | draft  54.1 ms/anh  -> nhanh 1.00x

chenh lech pixel tren 24 anh that (thang 0-255):
   sai lech LON NHAT   :    0.0
   sai lech trung binh :   0.00
   PSNR                :  138.1 dB  (thap nhat 138.1)
   -> anh te nhat van 138.1 dB > 45: duoi muc nhieu nen q90.
      Dat FAST_DECODE = True o C0 roi chay lai tu C0.

FAST_DECODE = False | JPEG_OPTIMIZE = False


## C4-ter · Vòng dựng shard

In [7]:
from tqdm.auto import tqdm

LO = 0 if SHARD_FROM is None else int(SHARD_FROM)
HI = N_SHARDS if SHARD_TO is None else int(SHARD_TO)
assert 0 <= LO < HI <= N_SHARDS, f'khoang [{LO},{HI}) khong hop le voi {N_SHARDS} shard'

# --- shard nao con thieu, va bao nhieu ANH thuc su phai lam ---
todo_sh = [i for i in range(LO, HI)
           if not os.path.exists(os.path.join(SHARD_DIR, f'lb_{i:05d}.tar'))]
n_todo  = int(work.shard.isin(todo_sh).sum())
_have   = len([f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')])
print(f'runtime nay: shard [{LO}, {HI}) / {N_SHARDS}  |  {_have} shard da co tren dia')
print(f'con phai dung: {len(todo_sh)} shard = {n_todo:,} anh')
if not todo_sh:
    print('-> khong con gi de lam, chay thang C5.')

t0, n_ok, all_errs, n_built = time.time(), 0, [], 0
pbar = tqdm(total=n_todo, unit='anh', unit_scale=True, smoothing=0.03,
            desc=f'letterbox {LB_SIZE}px', dynamic_ncols=True)
try:
    for i in todo_sh:
        st, n, errs = build_shard(i, pbar)
        n_ok += n; all_errs += errs
        if st == 'done':
            n_built += 1
            pbar.set_postfix_str(f'shard {n_built}/{len(todo_sh)} | loi {len(all_errs)} | '
                                 f'{(pbar.n*42*(LB_SIZE/518)**2/1e6):.1f} GB da ghi')
finally:
    pbar.close()
_el = time.time() - t0
print(f'\nxong. shard dung o runtime nay: {n_built} | anh ghi duoc: {n_ok:,} | '
      f'loi: {len(all_errs)} | {_el/3600:.2f} gio | {n_ok/max(_el,1e-9):.1f} img/s')
if all_errs:
    pd.DataFrame(all_errs, columns=['dicom_id', 'error']).to_csv(
        os.path.join(SHARD_DIR, '_errors.csv'), index=False)
    print('vai loi dau:', all_errs[:5])

runtime nay: shard [0, 378) / 378  |  11 shard da co tren dia
con phai dung: 367 shard = 366,110 anh


letterbox 1022px:   0%|          | 0.00/366k [00:00<?, ?anh/s]


xong. shard dung o runtime nay: 367 | anh ghi duoc: 366,110 | loi: 0 | 3.07 gio | 33.1 img/s


## C5 · Kiểm chứng + ghi manifest

In [8]:
missing_sh = [i for i in range(N_SHARDS)
              if not os.path.exists(os.path.join(SHARD_DIR, f'lb_{i:05d}.tar'))]
if missing_sh:
    print(f'!! thieu {len(missing_sh)} shard, vd {missing_sh[:8]}')
    print(f'   -> dat SHARD_FROM/SHARD_TO = ({min(missing_sh)}, {max(missing_sh)+1}) roi chay lai C4')
assert not missing_sh, f'thieu {len(missing_sh)} shard'

tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar'))
got = set()
for k, f in enumerate(tars, 1):
    with tarfile.open(os.path.join(SHARD_DIR, f)) as t:
        got |= {n[:-4] for n in t.getnames() if n.endswith('.jpg')}
    print(f'  doc {k}/{len(tars)} tar', end='\r')
tot = sum(os.path.getsize(os.path.join(SHARD_DIR, f)) for f in tars)
print(f'\n{len(tars)} tar | {len(got):,} anh | {tot/1e9:.2f} GB')

need = set(work.dicom_id)
miss = need - got
assert not miss, f'{len(miss):,} anh thieu (xem _errors.csv), vd {sorted(miss)[:5]}'
assert not (got - need), f'{len(got-need):,} anh la lot vao shard'
print(f'du ca {len(need):,} anh cua CXR-LT, khong du thua')

# --- anh trong shard phai dung LB_SIZE va co vien dem ---
with tarfile.open(os.path.join(SHARD_DIR, tars[0])) as t:
    for m in [m for m in t.getmembers() if m.name.endswith('.jpg')][:3]:
        im = Image.open(io.BytesIO(t.extractfile(m).read()))
        a = np.asarray(im)
        assert im.size == (LB_SIZE, LB_SIZE), im.size
        assert im.mode == ('L' if GRAYSCALE else 'RGB'), im.mode
        _rows = (a.max(axis=1) > 0).sum(); _cols = (a.max(axis=0) > 0).sum()
        print(f'  {m.name[:18]}...  {im.size} mode={im.mode}  '
              f'vung khac 0: {_cols}x{_rows}  (dem {LB_SIZE-_cols}x{LB_SIZE-_rows})')

WL = work[['dicom_id', 'study_id', 'subject_id', 'view_group', 'split', 'shard']]
WL.to_parquet(os.path.join(SHARD_DIR, '_work_list.parquet'), index=False)
json.dump(dict(mode='letterbox', store_size=LB_SIZE, model_size=MODEL_SIZE,
               grayscale=GRAYSCALE, jpeg_q=JPEG_Q, pad_value=PAD_VALUE,
               shard_size=SHARD_SIZE, n_shards=N_SHARDS, n_images=len(work),
               tar_prefix='lb_', nguon=DRIVE_IMG,
               nap_the_nao='KHONG CenterCrop — anh da o dung kich thuoc model',
               ghi_chu='giu 100% FOV; doi chung voi bo crop cua 04a'),
          open(os.path.join(SHARD_DIR, '_shard_meta.json'), 'w'), indent=1)
print(f'\nghi _work_list.parquet + _shard_meta.json')

print(f'''
================================================================
HAI BO SHARD SAN SANG CHO STAGE 3

  A. crop      cxr_560_shards/ + cxr_560_shards_cxrlt/   559px, CenterCrop(518)
  B. letterbox cxr_518_letterbox_cxrlt/                  518px, KHONG crop

31_cxrlt_embed.ipynb trich embedding CHO CA HAI, roi 32 so:
    CXR-only crop      mAP ?
    CXR-only letterbox mAP ?
  Chenh lech = gia that su cua 28.5% FOV ma nb19 do duoc tren KHUNG.
  (nb19 C7 — con chua chay — se noi bao nhieu trong so do la GIAI PHAU,
   bao nhieu chi la vien den collimation.)
================================================================''')


378 tar | 377,110 anh | 49.30 GB
du ca 377,110 anh cua CXR-LT, khong du thua
  02aa804e-bde0afdd-...  (1022, 1022) mode=L  vung khac 0: 787x848  (dem 235x174)
  174413ec-4ec4c1f7-...  (1022, 1022) mode=L  vung khac 0: 728x888  (dem 294x134)
  2a2277a9-b0ded155-...  (1022, 1022) mode=L  vung khac 0: 848x1022  (dem 174x0)

ghi _work_list.parquet + _shard_meta.json

HAI BO SHARD SAN SANG CHO STAGE 3

  A. crop      cxr_560_shards/ + cxr_560_shards_cxrlt/   559px, CenterCrop(518)
  B. letterbox cxr_518_letterbox_cxrlt/                  518px, KHONG crop

31_cxrlt_embed.ipynb trich embedding CHO CA HAI, roi 32 so:
    CXR-only crop      mAP ?
    CXR-only letterbox mAP ?
  Chenh lech = gia that su cua 28.5% FOV ma nb19 do duoc tren KHUNG.
  (nb19 C7 — con chua chay — se noi bao nhieu trong so do la GIAI PHAU,
   bao nhieu chi la vien den collimation.)


## C6 · Dọn bộ letterbox **518** cũ

Bộ 518 dựng dở trước đó giờ **thừa**: hạ mẫu từ 1022 xuống 518 lúc nạp là ra, mà 1022 còn giữ
được mọi mức trung gian. Cell này xoá nó để lấy lại chỗ.

Xoá là **không hoàn tác được**, nên cell in ra chính xác cái gì sẽ bị xoá và bao nhiêu GB trước
khi làm, và chỉ đụng vào **đúng một thư mục có tên khớp chính xác**.

In [11]:
DEL_DIR = os.path.join(DATA_DIR, 'cxr_518_letterbox_cxrlt')   # <<< CHI thu muc nay
DO_DELETE = True        # dat False neu chi muon xem se xoa gi

assert os.path.basename(DEL_DIR) == 'cxr_518_letterbox_cxrlt', 'ten thu muc khong khop -> DUNG'
assert os.path.abspath(DEL_DIR) != os.path.abspath(SHARD_DIR), 'trung thu muc dang dung -> DUNG'

if not os.path.isdir(DEL_DIR):
    print(f'khong co {DEL_DIR} -> khong can don')
else:
    fs = [os.path.join(DEL_DIR, f) for f in os.listdir(DEL_DIR)]
    tot = sum(os.path.getsize(f) for f in fs if os.path.isfile(f))
    ntar = len([f for f in fs if f.endswith('.tar')])
    print(f'{DEL_DIR}')
    print(f'  {len(fs)} file ({ntar} tar) | {tot/1e9:.1f} GB')
    if DO_DELETE:
        import shutil
        shutil.rmtree(DEL_DIR)
        print(f'  -> DA XOA, lay lai {tot/1e9:.1f} GB')
    else:
        print('  -> DO_DELETE=False, chua xoa gi')

print(f'''
Con lai tren dia:
  cxr_560_shards/ + cxr_560_shards_cxrlt/   crop 559px   -> arm A (CenterCrop 518)
  {os.path.basename(SHARD_DIR)}/            letterbox {LB_SIZE}px -> arm D, va ha xuong 518 ra arm B
''')

khong co /content/drive/MyDrive/MIMIC_CXR/planB_out/cxr_518_letterbox_cxrlt -> khong can don

Con lai tren dia:
  cxr_560_shards/ + cxr_560_shards_cxrlt/   crop 559px   -> arm A (CenterCrop 518)
  cxr_1022_letterbox_cxrlt/            letterbox 1022px -> arm D, va ha xuong 518 ra arm B

